In [ ]:
#@title Gemini Live Transcriber { display-mode: "form" }
# One run. Prompts: model -> API key -> upload -> automatic transcription.

import os, sys, re, math, time, json, shutil, asyncio, subprocess, getpass, zipfile
from array import array
from pathlib import Path
from datetime import datetime
from google.colab import files

SAMPLE_RATE = 16_000
BYTES_PER_SAMPLE = 2
BYTES_PER_SEC = SAMPLE_RATE * BYTES_PER_SAMPLE
FRAME_MS = 100
FRAME_BYTES = BYTES_PER_SEC * FRAME_MS // 1000
OVERLAP_SEC = 1.0
TMP = Path('/content/gemini_live_tmp')
TMP.mkdir(parents=True, exist_ok=True)

MODELS = {
    '1': {
        'name': 'Gemini 3.5 Transcribe Live',
        'id': 'gemini-3.5-transcribe-live',
        'tpm': 20_000,
        'concurrency': 12,
        'max_chunk_sec': 540,
    },
    '2': {
        'name': 'Gemini 3.8 Live',
        'id': 'gemini-3.8-live',
        'tpm': 65_000,
        'concurrency': 40,
        'max_chunk_sec': 480,
    },
}


def choose_models():
    print('Choose model:')
    print('  1 = Gemini 3.5 Transcribe Live')
    print('  2 = Gemini 3.8 Live')
    print('  3 = Both (3.5 first, then 3.8)')
    while True:
        choice = input('Model [1]: ').strip() or '1'
        if choice in ('1', '2', '3'):
            return ['1', '2'] if choice == '3' else [choice]
        print('Type 1, 2, or 3.')


def ask_key():
    while True:
        key = getpass.getpass('Gemini API key: ').strip()
        if key:
            return key
        print('API key cannot be empty.')


def upload_one_file():
    print('\nChoose the recording to upload...')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file was uploaded.')
    if len(uploaded) != 1:
        raise RuntimeError('Please upload exactly one recording.')
    name, data = next(iter(uploaded.items()))
    path = Path('/content') / Path(name).name
    path.write_bytes(data)
    return path


def install_sdk():
    print('\n[setup] Preparing Gemini SDK...', flush=True)
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'google-genai==2.25.0'],
        check=True,
    )
    global genai, types
    from google import genai as _genai
    from google.genai import types as _types
    genai, types = _genai, _types


def ffmpeg_to_pcm(src, dst):
    if not shutil.which('ffmpeg'):
        raise RuntimeError('ffmpeg is unavailable in this Colab runtime.')
    subprocess.run([
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
        '-i', str(src), '-vn', '-ac', '1', '-ar', str(SAMPLE_RATE),
        '-f', 's16le', str(dst),
    ], check=True)
    if not dst.exists() or dst.stat().st_size == 0:
        raise RuntimeError('Audio conversion failed or produced an empty file.')
    return dst.stat().st_size / BYTES_PER_SEC


def hms(sec):
    sec = max(0, int(round(sec)))
    h, rem = divmod(sec, 3600)
    m, s = divmod(rem, 60)
    return f'{h:02d}:{m:02d}:{s:02d}'


def safe_name(s):
    return re.sub(r'[^\w.-]+', '_', s, flags=re.UNICODE).strip('_') or 'transcript'


def make_chunks(total_sec, spec):
    c = min(spec['concurrency'], max(1, math.ceil(total_sec / 2)))
    waves = max(1, math.ceil(total_sec / (c * spec['max_chunk_sec'])))
    count = min(max(1, waves * c), max(1, math.ceil(total_sec / 2)))
    chunk_len = total_sec / count
    out = []
    for i in range(count):
        nominal_start = i * chunk_len
        nominal_end = total_sec if i == count - 1 else (i + 1) * chunk_len
        out.append({
            'index': i,
            'start_sec': max(0.0, nominal_start - (OVERLAP_SEC if i else 0.0)),
            'end_sec': min(total_sec, nominal_end + (OVERLAP_SEC if i < count - 1 else 0.0)),
        })
    return out


def norm_word(w):
    return re.sub(r'[^\w\u0600-\u06FF]+', '', w, flags=re.UNICODE).casefold()


def merge_overlap(parts, max_words=60):
    merged = []
    for text in parts:
        words = text.strip().split()
        if not words:
            continue
        if not merged:
            merged = words
            continue
        left = [norm_word(x) for x in merged]
        right = [norm_word(x) for x in words]
        best = 0
        for n in range(min(max_words, len(merged), len(words)), 1, -1):
            if left[-n:] == right[:n] and any(left[-n:]):
                best = n
                break
        merged.extend(words[best:])
    return ' '.join(merged).strip()


def probably_silent(path, start_byte, end_byte, threshold=120):
    span = max(0, end_byte - start_byte)
    if span <= 0:
        return True
    window = min(BYTES_PER_SEC // 2, span)
    positions = [start_byte] if span <= window else [
        start_byte + int((span - window) * i / 7) for i in range(8)
    ]
    peak = 0.0
    with open(path, 'rb') as f:
        for pos in positions:
            pos -= pos % 2
            f.seek(pos)
            data = f.read(window)
            if len(data) < 2:
                continue
            samples = array('h')
            samples.frombytes(data[:len(data) - (len(data) % 2)])
            if sys.byteorder != 'little':
                samples.byteswap()
            if samples:
                rms = math.sqrt(sum(v * v for v in samples) / len(samples))
                peak = max(peak, rms)
    return peak < threshold


def model_config(key):
    if key == '1':
        return {
            'response_modalities': ['TEXT'],
            'input_audio_transcription': {
                'language_codes': [],
                'mode': 'VERBATIM',
            },
        }
    return {
        'response_modalities': ['AUDIO'],
        'input_audio_transcription': {},
    }


async def receive_transcript(session, stream_done):
    pieces = []
    iterator = session.receive().__aiter__()
    last_text_at = time.monotonic()
    while True:
        timeout = 30.0 if not stream_done.is_set() else (4.0 if pieces else 20.0)
        try:
            msg = await asyncio.wait_for(iterator.__anext__(), timeout=timeout)
        except (StopAsyncIteration, asyncio.TimeoutError):
            break
        sc = getattr(msg, 'server_content', None)
        tr = getattr(sc, 'input_transcription', None) if sc else None
        text = (getattr(tr, 'text', '') or '').strip() if tr else ''
        if text:
            if not pieces or text != pieces[-1]:
                pieces.append(text)
            last_text_at = time.monotonic()
        if stream_done.is_set() and pieces and time.monotonic() - last_text_at > 2.0:
            break
    return ' '.join(pieces).strip()


async def transcribe_chunk(client, pcm_path, chunk, model_key, max_attempts=3):
    spec = MODELS[model_key]
    start_byte = int(chunk['start_sec'] * BYTES_PER_SEC)
    end_byte = int(chunk['end_sec'] * BYTES_PER_SEC)
    start_byte -= start_byte % 2
    end_byte -= end_byte % 2
    total = max(0, end_byte - start_byte)
    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            async with client.aio.live.connect(model=spec['id'], config=model_config(model_key)) as session:
                done = asyncio.Event()
                recv = asyncio.create_task(receive_transcript(session, done))
                remaining = total
                with open(pcm_path, 'rb', buffering=1024 * 1024) as f:
                    f.seek(start_byte)
                    next_send = time.monotonic()
                    while remaining > 0:
                        data = f.read(min(FRAME_BYTES, remaining))
                        if not data:
                            break
                        await session.send_realtime_input(
                            audio=types.Blob(data=data, mime_type='audio/pcm;rate=16000')
                        )
                        remaining -= len(data)
                        next_send += FRAME_MS / 1000.0
                        delay = next_send - time.monotonic()
                        if delay > 0:
                            await asyncio.sleep(delay)
                await session.send_realtime_input(audio_stream_end=True)
                done.set()
                try:
                    text = await asyncio.wait_for(recv, timeout=30.0)
                finally:
                    if not recv.done():
                        recv.cancel()
                if not text and not await asyncio.to_thread(
                    probably_silent, pcm_path, start_byte, end_byte
                ):
                    raise RuntimeError('empty transcript on non-silent audio')
                return text
        except Exception as e:
            last_error = e
            if attempt < max_attempts:
                await asyncio.sleep(2 * attempt)
    raise RuntimeError(f'chunk {chunk["index"] + 1} failed: {last_error}')


async def transcribe_model(api_key, pcm_path, duration, model_key):
    spec = MODELS[model_key]
    chunks = make_chunks(duration, spec)
    concurrency = min(spec['concurrency'], len(chunks))
    results = [None] * len(chunks)
    client = genai.Client(api_key=api_key)
    started = time.monotonic()
    done_count = 0
    stop_heartbeat = asyncio.Event()

    expected = max(ch['end_sec'] - ch['start_sec'] for ch in chunks) * math.ceil(len(chunks) / concurrency)
    print(
        f'\n[{spec["name"]}] {len(chunks)} chunks | up to {concurrency} parallel | '
        f'audio {hms(duration)} | rough minimum ~{expected/60:.1f} min',
        flush=True,
    )

    async def heartbeat():
        while not stop_heartbeat.is_set():
            elapsed = time.monotonic() - started
            print(
                f'\r[{spec["name"]}] working... {done_count}/{len(chunks)} chunks | elapsed {hms(elapsed)}',
                end='', flush=True,
            )
            try:
                await asyncio.wait_for(stop_heartbeat.wait(), timeout=10)
            except asyncio.TimeoutError:
                pass

    hb = asyncio.create_task(heartbeat())

    async def worker(ch):
        nonlocal done_count
        text = await transcribe_chunk(client, pcm_path, ch, model_key)
        results[ch['index']] = text
        done_count += 1

    async def run_wave(wave):
        tasks = [asyncio.create_task(worker(ch)) for ch in wave]
        settled = await asyncio.gather(*tasks, return_exceptions=True)
        return [x for x in settled if isinstance(x, Exception)]

    try:
        failures = []
        for offset in range(0, len(chunks), concurrency):
            failures.extend(await run_wave(chunks[offset:offset + concurrency]))

        missing = [ch for ch in chunks if results[ch['index']] is None]
        recovery = max(1, concurrency // 2)
        round_no = 0
        while missing and round_no < 5:
            round_no += 1
            print(f'\n[{spec["name"]}] retrying {len(missing)} failed chunk(s) with concurrency {recovery}...', flush=True)
            for offset in range(0, len(missing), recovery):
                await run_wave(missing[offset:offset + recovery])
            missing = [ch for ch in chunks if results[ch['index']] is None]
            recovery = max(1, recovery // 2)

        if missing:
            raise RuntimeError(
                f'{len(missing)} chunk(s) still failed after retries; no incomplete transcript was saved.'
            )

        transcript = merge_overlap(results)
        elapsed = time.monotonic() - started
        print(f'\r[{spec["name"]}] done: {len(chunks)}/{len(chunks)} chunks | {elapsed/60:.2f} min' + ' ' * 20, flush=True)
        return transcript, elapsed
    finally:
        stop_heartbeat.set()
        try:
            await hb
        except Exception:
            pass
        try:
            client.close()
        except Exception:
            pass


async def main():
    chosen = choose_models()
    api_key = ask_key()
    src = upload_one_file()
    install_sdk()

    pcm = TMP / f'{int(time.time())}_{safe_name(src.stem)}.pcm'
    print('\n[1/3] Preparing audio...', flush=True)
    prep_start = time.monotonic()
    duration = await asyncio.to_thread(ffmpeg_to_pcm, src, pcm)
    print(f'[1/3] Ready: {src.name} | duration {hms(duration)} | prep {time.monotonic()-prep_start:.1f}s', flush=True)

    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    base = safe_name(src.stem)
    outputs = []
    timings = []

    try:
        print('[2/3] Transcribing...', flush=True)
        for key in chosen:
            text, elapsed = await transcribe_model(api_key, pcm, duration, key)
            spec = MODELS[key]
            out = Path('/content') / f'{base}_{spec["id"]}_{stamp}.txt'
            out.write_text(text, encoding='utf-8')
            outputs.append(out)
            timings.append((spec['name'], elapsed))

        print('\n[3/3] Finished.', flush=True)
        for name, elapsed in timings:
            print(f'  {name}: {elapsed/60:.2f} min')

        if len(outputs) == 1:
            print(f'\nDownloading: {outputs[0].name}', flush=True)
            files.download(str(outputs[0]))
        else:
            z = Path('/content') / f'{base}_Gemini_Live_transcripts_{stamp}.zip'
            with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as archive:
                for p in outputs:
                    archive.write(p, arcname=p.name)
            print(f'\nDownloading: {z.name}', flush=True)
            files.download(str(z))
    finally:
        try:
            pcm.unlink(missing_ok=True)
        except Exception:
            pass


await main()


In [ ]:
#@title Gemini Live Concurrency Benchmark { display-mode: "form" }
# One run. Prompts: model -> API key -> upload. Tests 12/6/3 concurrent Live sessions on a 30s speech probe.

import os, sys, re, math, time, asyncio, subprocess, getpass, shutil
from array import array
from pathlib import Path
from collections import Counter
from google.colab import files

SAMPLE_RATE = 16_000
BYTES_PER_SAMPLE = 2
BYTES_PER_SEC = SAMPLE_RATE * BYTES_PER_SAMPLE
FRAME_MS = 100
FRAME_BYTES = BYTES_PER_SEC * FRAME_MS // 1000
PROBE_SEC = 30
JOBS = 12
PROFILES = (3, 6, 12)
TMP = Path('/content/gemini_live_bench_tmp')
TMP.mkdir(parents=True, exist_ok=True)

MODELS = {
    '1': ('Gemini 3.5 Transcribe Live', 'gemini-3.5-transcribe-live'),
    '2': ('Gemini 3.8 Live', 'gemini-3.8-live'),
}


def choose_models():
    print('Benchmark model:')
    print('  1 = Gemini 3.5 Transcribe Live')
    print('  2 = Gemini 3.8 Live')
    print('  3 = Both')
    while True:
        choice = input('Model [1]: ').strip() or '1'
        if choice in ('1', '2', '3'):
            return ['1', '2'] if choice == '3' else [choice]
        print('Type 1, 2, or 3.')


def ask_key():
    while True:
        key = getpass.getpass('Gemini API key: ').strip()
        if key:
            return key
        print('API key cannot be empty.')


def upload_one_file():
    print('\nChoose the recording to upload...')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file was uploaded.')
    if len(uploaded) != 1:
        raise RuntimeError('Please upload exactly one recording.')
    name, data = next(iter(uploaded.items()))
    path = Path('/content') / Path(name).name
    path.write_bytes(data)
    return path


def install_sdk():
    print('\n[setup] Preparing Gemini SDK...', flush=True)
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'google-genai==2.25.0'],
        check=True,
    )
    global genai, types
    from google import genai as _genai
    from google.genai import types as _types
    genai, types = _genai, _types


def ffmpeg_to_pcm(src, dst):
    if not shutil.which('ffmpeg'):
        raise RuntimeError('ffmpeg is unavailable in this Colab runtime.')
    subprocess.run([
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
        '-i', str(src), '-vn', '-ac', '1', '-ar', str(SAMPLE_RATE),
        '-f', 's16le', str(dst),
    ], check=True)
    if not dst.exists() or dst.stat().st_size == 0:
        raise RuntimeError('Audio conversion failed or produced an empty file.')
    return dst.stat().st_size / BYTES_PER_SEC


def model_config(key):
    if key == '1':
        return {
            'response_modalities': ['TEXT'],
            'input_audio_transcription': {'language_codes': [], 'mode': 'VERBATIM'},
        }
    return {'response_modalities': ['AUDIO'], 'input_audio_transcription': {}}


def rms_at(path, start_sec, span_sec):
    start = int(start_sec * BYTES_PER_SEC) // 2 * 2
    size = int(span_sec * BYTES_PER_SEC) // 2 * 2
    with open(path, 'rb') as f:
        f.seek(start)
        data = f.read(size)
    samples = array('h')
    samples.frombytes(data[:len(data) - (len(data) % 2)])
    if sys.byteorder != 'little':
        samples.byteswap()
    if not samples:
        return 0.0
    return math.sqrt(sum(v * v for v in samples) / len(samples))


def choose_probe(pcm_path, duration):
    probe = min(float(PROBE_SEC), duration)
    if duration <= probe:
        return 0.0, probe
    max_start = duration - probe
    starts = [max_start * p for p in (0.10, 0.30, 0.50, 0.70, 0.90)]
    scored = [(rms_at(pcm_path, s, probe), s) for s in starts]
    _, start = max(scored)
    return start, probe


async def receive_transcript(session, stream_done):
    pieces = []
    iterator = session.receive().__aiter__()
    last_text_at = time.monotonic()
    while True:
        timeout = 20.0 if not stream_done.is_set() else (4.0 if pieces else 15.0)
        try:
            msg = await asyncio.wait_for(iterator.__anext__(), timeout=timeout)
        except (StopAsyncIteration, asyncio.TimeoutError):
            break
        sc = getattr(msg, 'server_content', None)
        tr = getattr(sc, 'input_transcription', None) if sc else None
        text = (getattr(tr, 'text', '') or '').strip() if tr else ''
        if text and (not pieces or text != pieces[-1]):
            pieces.append(text)
            last_text_at = time.monotonic()
        if stream_done.is_set() and pieces and time.monotonic() - last_text_at > 2.0:
            break
    return ' '.join(pieces).strip()


async def one_probe(client, pcm_path, start_sec, probe_sec, model_key):
    model_name, model_id = MODELS[model_key]
    start_byte = int(start_sec * BYTES_PER_SEC) // 2 * 2
    total = int(probe_sec * BYTES_PER_SEC) // 2 * 2
    t0 = time.monotonic()
    recv = None
    try:
        async with client.aio.live.connect(model=model_id, config=model_config(model_key)) as session:
            done = asyncio.Event()
            recv = asyncio.create_task(receive_transcript(session, done))
            remaining = total
            with open(pcm_path, 'rb', buffering=1024 * 1024) as f:
                f.seek(start_byte)
                next_send = time.monotonic()
                while remaining > 0:
                    data = f.read(min(FRAME_BYTES, remaining))
                    if not data:
                        break
                    await session.send_realtime_input(
                        audio=types.Blob(data=data, mime_type='audio/pcm;rate=16000')
                    )
                    remaining -= len(data)
                    next_send += FRAME_MS / 1000.0
                    delay = next_send - time.monotonic()
                    if delay > 0:
                        await asyncio.sleep(delay)
            await session.send_realtime_input(audio_stream_end=True)
            done.set()
            try:
                text = await asyncio.wait_for(recv, timeout=20.0)
            finally:
                if not recv.done():
                    recv.cancel()
            if not text:
                raise RuntimeError('empty transcript')
            return {'ok': True, 'elapsed': time.monotonic() - t0, 'error': ''}
    except Exception as e:
        return {
            'ok': False,
            'elapsed': time.monotonic() - t0,
            'error': f'{type(e).__name__}: {e}',
        }
    finally:
        if recv is not None and not recv.done():
            recv.cancel()


async def run_profile(client, pcm_path, start_sec, probe_sec, model_key, concurrency):
    sem = asyncio.Semaphore(concurrency)

    async def job():
        async with sem:
            return await one_probe(client, pcm_path, start_sec, probe_sec, model_key)

    t0 = time.monotonic()
    results = await asyncio.gather(*(job() for _ in range(JOBS)))
    elapsed = time.monotonic() - t0
    success = sum(r['ok'] for r in results)
    errors = Counter(r['error'] for r in results if not r['ok'])
    total_audio = JOBS * probe_sec
    rtf = total_audio / elapsed if elapsed else float('inf')
    return success, elapsed, rtf, errors


async def benchmark_model(api_key, pcm_path, duration, model_key):
    model_name, _ = MODELS[model_key]
    start_sec, probe_sec = choose_probe(pcm_path, duration)
    print(
        f'\n[{model_name}] probe={probe_sec:.0f}s | jobs={JOBS} | '
        f'source offset={start_sec:.1f}s | profiles={"/".join(map(str, PROFILES))}',
        flush=True,
    )
    client = genai.Client(api_key=api_key)
    rows = []
    try:
        for concurrency in PROFILES:
            print(f'  testing concurrency {concurrency}...', flush=True)
            success, elapsed, rtf, errors = await run_profile(
                client, pcm_path, start_sec, probe_sec, model_key, concurrency
            )
            top_error = errors.most_common(1)[0][0] if errors else ''
            rows.append((concurrency, success, JOBS - success, elapsed, rtf, top_error))
    finally:
        try:
            client.close()
        except Exception:
            pass

    print('\n  concurrency | success | failed | elapsed | effective speed | top error')
    print('  ------------+---------+--------+---------+-----------------+----------')
    for c, ok, fail, elapsed, rtf, err in rows:
        short_err = (err[:120] + '...') if len(err) > 123 else err
        print(f'  {c:>11} | {ok:>2}/{JOBS:<2}   | {fail:>6} | {elapsed:>6.1f}s | {rtf:>7.2f}x realtime | {short_err}')

    stable = [row for row in rows if row[1] == JOBS]
    if stable:
        best = max(stable, key=lambda row: row[0])
        print(f'\n  Highest fully successful tested concurrency: {best[0]}')
    else:
        print('\n  No tested concurrency achieved 100% success. Inspect the printed errors above.')


async def main():
    chosen = choose_models()
    api_key = ask_key()
    src = upload_one_file()
    install_sdk()

    pcm = TMP / f'{int(time.time())}_benchmark.pcm'
    try:
        print('\nPreparing audio...', flush=True)
        duration = await asyncio.to_thread(ffmpeg_to_pcm, src, pcm)
        print(f'Ready: {src.name} | duration {duration/60:.2f} min', flush=True)
        for key in chosen:
            await benchmark_model(api_key, pcm, duration, key)
    finally:
        try:
            pcm.unlink(missing_ok=True)
        except Exception:
            pass


await main()
